# HomeMatch — a personalized real estate agent

A RAG application that turns a buyer's plain-language wishlist into personalized
property recommendations.

1. **Generate** a synthetic listing catalogue with an LLM and a strict Pydantic schema.
2. **Store** the listings as embeddings in a Chroma vector database.
3. **Retrieve** the closest matches to the buyer's stated preferences.
4. **Personalize** each retrieved listing's description to emphasize what *this*
   buyer said they cared about — without inventing facts about the property.


## Setup


In [ ]:
!pip uninstall -y langchain-core langchain-community langchain-classic
!pip install -q --user "pydantic<2"
!pip install -q --user "numpy<2"
!pip install -q --user "langsmith<0.1"
!pip install -q --user "chromadb==0.4.12"
!pip install -q --user openai pandas

In [6]:
import os
import pandas as pd
import shutil
from typing import List
from fastapi.encoders import jsonable_encoder

# Chain libs
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.evaluation import load_evaluator
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.vectorstores.chroma import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, NonNegativeInt
from langchain.prompts import PromptTemplate

from dotenv import load_dotenv

# Credentials come from the environment, never from source.
# Copy .env.example to .env and fill in your key (.env is gitignored).
load_dotenv()
os.environ.setdefault("OPENAI_API_BASE", "https://api.openai.com/v1")

# Read back from os.environ so that LangChain helpers which pick the
# credentials up implicitly (e.g. OpenAIEmbeddings()) see the same values.
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]
model = "gpt-3.5-turbo"

# Listing Generation

In [20]:
# load the model
llm = OpenAI(model_name=model, temperature=0, api_key=OPENAI_API_KEY)

INSTRUCTION = "Generate eleven realistic real estate listings from diverse neighborhoods."
SAMPLE_LISTING = """
Here's a sample listing:

Neighborhood: Green Oaks
Price ($): 800,000
Bedrooms: 3
Bathrooms: 2
House Size (sqft): 2,000
Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.
Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze.
"""

class RealEstateListing(BaseModel):
    """
    A real estate listing.
    
    Attributes:
    - neighborhood: str
    - price: NonNegativeInt
    - bedrooms: NonNegativeInt
    - bathrooms: NonNegativeInt
    - house_size: NonNegativeInt
    - description: str
    - neighborhood_description: str
    """
    neighborhood: str = Field(description="The neighborhood where the listing is located")
    price: NonNegativeInt = Field(description="The price of the property in USD")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms in the property")
    bathrooms: NonNegativeInt = Field(description="Number of bathrooms in the property")
    house_size: NonNegativeInt = Field(description="Property size in square feet")
    description: str = Field(description="A short description of the property")
        
class ListingCollection(BaseModel):
    listing: List[RealEstateListing] = Field(description="List of available real estate")
   

/opt/conda/lib/python3.10/site-packages/langchain/llms/openai.py:202: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/langchain/llms/openai.py:790: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


In [21]:
parser = PydanticOutputParser(pydantic_object=ListingCollection)

In [22]:
prompt = PromptTemplate(
    template="{instruction}\n{sample}\n{format_instructions}\n",
    input_variables=["instruction", "sample"],
    partial_variables={"format_instructions": parser.get_format_instructions},
)
query = prompt.format(
    instruction=INSTRUCTION,
    sample=SAMPLE_LISTING,
)
print(query)

Generate eleven realistic real estate listings from diverse neighborhoods.

Here's a sample listing:

Neighborhood: Green Oaks
Price ($): 800,000
Bedrooms: 3
Bathrooms: 2
House Size (sqft): 2,000
Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.
Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. Wit

In [23]:
resp = llm(query)

In [24]:
result = parser.parse(resp)
df = pd.DataFrame(jsonable_encoder(result.listing))
df.head()
df.to_csv('listings.csv', index_label = 'id')

# Storing Listings in a Vector Database

In [28]:
csv_path= 'listings.csv'

embedding_function = OpenAIEmbeddings()

documents = []
df = pd.read_csv(csv_path)
for index, row in df.iterrows():
    documents.append(Document(page_content=row['description'], metadata={'id': str(index)}))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

if chunks:
    document = chunks[10]
    print(document.page_content)
    print(document.metadata)
    
if os.path.exists("chroma"):
    shutil.rmtree("chroma")

db = Chroma.from_documents(
    chunks, OpenAIEmbeddings(), persist_directory="chroma"
)
db.persist()
print(f"Saved {len(chunks)} chunks to {'chroma'}.")

Split 11 documents into 23 chunks.
with a spa. The interior features high-end finishes, a gourmet kitchen, and a home theater. Enjoy resort-style living in this exclusive waterfront property.
{'id': '4', 'start_index': 197}
Saved 23 chunks to chroma.


# Building the User Preference Interface

In [32]:
query_text = "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system." 

prompt_template = """
Within the following context: {context}
---
Answer this question : {question}
"""

# Search and Augmented Response Gen

In [35]:
def predict_response(query_text, PROMPT_TEMPLATE):
    embedding_function = OpenAIEmbeddings()
    db = Chroma(persist_directory="chroma", embedding_function=embedding_function)

    # Search the DB.
    results = db.similarity_search_with_relevance_scores(query_text, k=3)
    if len(results) == 0 or results[0][1] < 0.7:
        print(f"Unable to find matching results.")
    else:
        context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
        prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
        prompt = prompt_template.format(context=context_text, question=query_text)
        print(f"Generated Prompt:\n{prompt}")
        
        model = ChatOpenAI()
        response_text = model.predict(prompt)
        sources = [doc.metadata.get("id", None) for doc, _score in results]
        formatted_response = f"Response: {response_text}\nSources: {sources}"
        print(formatted_response)

In [36]:
predict_response(query_text, prompt_template)

Generated Prompt:
Human: 
Within the following context: living areas are perfect for entertaining, while the backyard offers a peaceful retreat with a garden and patio. Embrace the charm and character of this historic gem.

---

Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious

---

fireplace, updated kitchen, and a master suite with a walk-in closet. Enjoy the peace and quiet of suburban living while still being close to schools, parks, and shopping.
---
Answer this question : A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.

Response: What are some additional features of this eco-friendly oasis in Green Oaks?
Sources: ['3', '0', '2']


# Personalizing Listing Descriptions

In [37]:
AUGMENTED_PROMPT_TEMPLATE ="""
Based on the following context:

{context}

---

Answer the question {question} with a clear, engaging explanation that subtly highlights property features aligned with the buyer’s preferences.
"""

In [38]:
predict_response(query_text, AUGMENTED_PROMPT_TEMPLATE)

Generated Prompt:
Human: 
Based on the following context:

living areas are perfect for entertaining, while the backyard offers a peaceful retreat with a garden and patio. Embrace the charm and character of this historic gem.

---

Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious

---

fireplace, updated kitchen, and a master suite with a walk-in closet. Enjoy the peace and quiet of suburban living while still being close to schools, parks, and shopping.

---

Answer the question A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system. with a clear, engaging explanation that subtly highlights property features aligned with the buyer’s preferences.

Response: This home in Green Oaks offers a backyard perfect